# Mexico — PSTS Self-Employment: Pre-Trends & Control DiD

Outcome throughout: `log(SelfEmployedPSTS / LaborForce)` by state × year × gender.
PSTS self-employed = the all-PSTS total (employers + own-account persons),
matching Canada's StatCan "Self-employed" series.

Three steps:
1. **Within-Mexico pre-trends** (2014–2017) — male vs female.
2. **Cross-country DDD pre-trends** (2014–2017) — does the female–male gap trend in parallel in Canada vs Mexico.
3. **Mexico control DiD** (2014–2024) — the Canadian DiD spec run on Mexico (no WES); used to check for a *common* post-2018 gender shock (see that section).

In [ ]:
# Data preparation (within-Mexico)
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

sns.set_theme(style="whitegrid")

# --- Locate the repo root, independent of the kernel's working directory ------
# VS Code starts the Jupyter kernel in the workspace root, while `jupyter`
# launched from mexico/ starts it in the notebook folder. Anchor every path to
# the repo root (the folder that holds both data/ and WESEffectsDiD/) so the
# notebook reads/writes the same files no matter where the kernel starts.
def _find_root(markers=('data', 'WESEffectsDiD')):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if all((cand / m).exists() for m in markers):
            return cand
    raise FileNotFoundError(
        'Could not locate the repo root (expected folders data/ and '
        f'WESEffectsDiD/). Searched upward from {here}.')

ROOT = _find_root()
RESULTS = ROOT / 'mexico' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Shared styled-table renderer (blue header banner + unchanged summary body).
sys.path.insert(0, str(ROOT))
from result_table_style import save_result_table

# --- Harmonized gender recode -------------------------------------------------
# The country datasets label sex differently: StatCan (Canada) uses 'Men+'/'Women+',
# while the US Census and INEGI (Mexico) use 'Male'/'Female'. Map all variants onto a
# single binary `female` indicator (and a canonical 'Male'/'Female' label) so the
# sources stack cleanly in the pooled cross-country analysis below.
SEX_TO_FEMALE = {'Women+': 1, 'Women': 1, 'Female': 1,
                 'Men+': 0,  'Men': 0,   'Male': 0}

def recode_sex(s):
    """Map a raw Sex label series onto the {0 = male, 1 = female} indicator."""
    out = s.astype(str).str.strip().map(SEX_TO_FEMALE)
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).str.strip().unique())
        raise ValueError(f"Unrecognised Sex label(s): {bad}")
    return out.astype(int)

# Mexico data is now long (one row per State x Year x Sex), mirroring the Canadian
# StatCan layout. Read it directly and recode the Sex column.
df = pd.read_csv(ROOT / 'data' / 'mexico' / 'aggregatedData.csv').rename(columns={
    'State': 'state', 'Year': 'year',
    'SelfEmployedPSTS': 'psts_self_employed', 'LaborForce': 'labor_force'})
df['female'] = recode_sex(df['Sex'])
df['gender'] = df['female'].map({0: 'Male', 1: 'Female'})
df['rate']     = df['psts_self_employed'] / df['labor_force']
df['log_rate'] = np.log(df['rate'])

# Pre-treatment window 2014-2017.
df_final = df[df['year'].between(2014, 2017)].copy()
df_final['year'] = df_final['year'].astype(int)

print('Pre-treatment rows 2014-2017:', len(df_final),
      '| states:', df_final['state'].nunique(),
      '| years:', sorted(df_final['year'].unique()))
df_final.head()

## 1. Within-Mexico parallel trends (male vs female)

`log_rate ~ female * C(year) + C(state)`, clustered by state. Treatment = `female`;
joint F-test on the `female:C(year)` interactions. A non-significant F means no
evidence against parallel pre-trends.

In [ ]:
# Within-Mexico parallel trends + OLS + F-test (All PSTS)
df_pre = df_final.dropna(subset=['log_rate', 'labor_force']).copy()  # OLS/F-test sample: 2014-2017

# Full-range frame for the trajectory plot (OLS/F-test stay on 2014-2017).
df_plot = df.dropna(subset=['log_rate', 'labor_force']).copy()
df_plot['year'] = df_plot['year'].astype(int)

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot, x='year', y='log_rate', hue='gender', marker='o', linewidth=2.5)
plt.axvline(x=2018, color='red', linestyle='--', label='Treatment cutoff (2018)')
plt.title('Mexico Parallel Trends: All PSTS self-employed (log Rate)', fontsize=14)
plt.ylabel('Log(PSTS self-employed / Labor Force)')
plt.xticks(sorted(df_plot['year'].unique()))
plt.legend()
plt.savefig(RESULTS / 'mexico_parallelTrends_psts_all.png', dpi=300, bbox_inches='tight')
plt.show()

model = smf.ols("log_rate ~ female * C(year) + C(state)", data=df_pre)
res = model.fit(cov_type='cluster', cov_kwds={'groups': df_pre['state']})
print(res.summary())

interaction_vars = [v for v in res.params.index if 'female:C(year)' in v]
f_test = res.f_test(interaction_vars)
print('\n--- F-test on female:C(year) interactions (All PSTS) ---')
print(f_test.summary())

# Styled result graphics (blue header banner + unchanged summary body).
save_result_table('OLS Regression: Mexican Parallel Trends (2014–2017)',
                  res.summary(), RESULTS / 'Mexico_ParallelTrends_OLS_allPSTS.png')
save_result_table('Joint F-Test: Mexican Pre-Treatment Interactions',
                  f_test.summary(), RESULTS / 'Mexico_FTest_allPSTS.png')

## 2. Cross-country pre-trends: Canada vs Mexico (DDD)

Mexico is the counterfactual for Canada (WES enacted 2018). The WES targets *women*,
so the quantity that must trend in parallel across countries is the **gender gap** —
women's PSTS self-employment relative to men's. Pooling both genders and both
countries on 2014–2017:

    log_rate ~ C(geo) + C(year) + female + female:C(year)
               + canada:C(year) + female:canada + female:canada:C(year)

clustered by `geo`. Parallel-trends object = `female:canada:C(year)`; a non-significant
joint F means no evidence the gender gap diverged across countries before 2018.

Comparable outcome: Mexico `SelfEmployedPSTS` (employers + own-account) ↔ Canada
"Self-employed" in PSTS (StatCan 14-10-0027-01) — both total self-employed persons
in NAICS-54, as a rate.

In [ ]:
# ---- Shared data prep: pool Canada + Mexico (both genders) for the DDD ----
sns.set_theme(style="whitegrid")

# Canada: self-employed in PSTS / labour force (StatCan 14-10-0027-01 + LFS controls)
ca_se = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedSelfEmployed.csv')
ca_lf = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedLabourStats.csv')
ca = pd.merge(ca_se,
              ca_lf[['Province', 'Year', 'Sex', 'Control_LaborForce']],
              on=['Province', 'Year', 'Sex'], how='left')
ca['female'] = recode_sex(ca['Sex'])          # harmonize StatCan 'Men+'/'Women+'
ca = ca.rename(columns={'Province': 'geo', 'Year': 'year',
                        'Self_Employed': 'num', 'Control_LaborForce': 'lf'})
ca['country'] = 'Canada'
ca['canada'] = 1
# Canadian source has only the aggregate "Self-employed" class (no employer/
# own-account split) -> the Canada-equivalent of Mexico's all-PSTS total.
ca_base = ca[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf']].copy()

# Mexico: now long (one row per State x Year x Sex); recode the same way.
mx = pd.read_csv(ROOT / 'data' / 'mexico' / 'aggregatedData.csv')

def _mexico_long():
    """Mexico (long) -> pooled schema for the all-PSTS self-employed numerator."""
    out = mx.rename(columns={'State': 'geo', 'Year': 'year',
                             'SelfEmployedPSTS': 'num', 'LaborForce': 'lf'}).copy()
    out['female'] = recode_sex(out['Sex'])     # harmonize INEGI 'Male'/'Female'
    out['country'] = 'Mexico'; out['canada'] = 0
    return out[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf']]

def build_pooled():
    """Pooled Canada+Mexico long frame, pre-treatment 2014-2017, with log-rate outcome."""
    pooled = pd.concat([ca_base, _mexico_long()], ignore_index=True)
    pooled['geo'] = pooled['country'] + ':' + pooled['geo'].astype(str)  # unique geo ids
    pooled = pooled[pooled['year'].between(2014, 2017)].copy()
    pooled['year'] = pooled['year'].astype(int)
    pooled['rate'] = pooled['num'] / pooled['lf']
    pooled['log_rate'] = np.log(pooled['rate'])
    pooled = pooled.dropna(subset=['log_rate', 'lf'])
    pooled = pooled[np.isfinite(pooled['log_rate'])].copy()
    return pooled

# DDD pre-trends specification. The geo fixed effects absorb the country main
# effect; the differential gender-gap *trend* female:canada:C(year) is the test.
DDD_FORMULA = ("log_rate ~ C(geo) + C(year) + female + female:C(year) "
               "+ canada:C(year) + female:canada + female:canada:C(year)")

def fit_ddd(pooled):
    res = smf.ols(DDD_FORMULA, data=pooled).fit(
        cov_type='cluster', cov_kwds={'groups': pooled['geo']})
    triple = [v for v in res.params.index
              if ('female' in v) and ('canada' in v) and ('C(year)' in v)]
    return res, res.f_test(triple), triple

def ddd_plot(pooled, title, outfile):
    """Left: mean log-rate by country x gender. Right: gender gap (women-men) per country."""
    p = pooled.copy()
    p['Gender'] = p['female'].map({0: 'Men', 1: 'Women'})
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.lineplot(ax=axes[0], data=p, x='year', y='log_rate', hue='country',
                 style='Gender', markers=True, dashes=True, linewidth=2.2, errorbar=None)
    axes[0].set_title('Mean log(PSTS self-employed / labour force)')
    axes[0].set_ylabel('Log rate'); axes[0].set_xticks(sorted(p['year'].unique()))
    gap = (p.groupby(['country', 'year', 'female'])['log_rate'].mean().unstack('female'))
    gap['gap'] = gap[1] - gap[0]
    gap = gap.reset_index()
    sns.lineplot(ax=axes[1], data=gap, x='year', y='gap', hue='country',
                 marker='o', linewidth=2.6)
    axes[1].set_title('Gender gap = log(women) - log(men)  [DDD object]')
    axes[1].set_ylabel('Female - Male (log points)'); axes[1].set_xticks(sorted(p['year'].unique()))
    fig.suptitle(title, fontsize=14); fig.tight_layout()
    fig.savefig(outfile, dpi=300, bbox_inches='tight'); plt.show()

print('Canada rows:', len(ca_base), '| Mexico states:', mx['State'].nunique())

In [ ]:
# Cross-country DDD (All PSTS): Mexico SelfEmployedPSTS vs Canada self-employed
pooled = build_pooled()
print('Pooled rows:', len(pooled),
      '| clusters (geo):', pooled['geo'].nunique(),
      '| Canada:', int((pooled.country == 'Canada').sum()),
      '| Mexico:', int((pooled.country == 'Mexico').sum()),
      '| years:', sorted(pooled['year'].unique()))

ddd_plot(pooled,
         'Canada vs Mexico - Cross-Country Pre-Trends (All PSTS), 2014-2017',
         RESULTS / 'canadaMexico_ddd_parallelTrends_allPSTS.png')

res_ddd, ftest_ddd, triple_ddd = fit_ddd(pooled)
print(res_ddd.summary())
print('\n--- DDD F-test on female:canada:C(year) interactions (All PSTS) ---')
print('Tested terms:', triple_ddd)
print(ftest_ddd.summary())

# Styled result graphics (blue header banner + unchanged summary body).
save_result_table('OLS Regression: Cross-Country (DDD) Pre-Trends (2014–2017)',
                  res_ddd.summary(), RESULTS / 'CanadaMexico_DDD_OLS_allPSTS.png')
save_result_table('Joint F-Test: Cross-Country (DDD) Pre-Treatment Interactions',
                  ftest_ddd.summary(), RESULTS / 'CanadaMexico_DDD_FTest_allPSTS.png')

## 3. Control DiD (Mexico) — common-shock check

Canadian DiD specification (`WESEffectsDiD/Main.ipynb`) run on Mexico:

    log_rate ~ Treat + Treat:Post + UnemploymentRate + C(state) + C(year)

`Treat` = `female`, `Post` = `year ≥ 2018` (absorbed by `C(year)`); clustered by state, 2014–2024.

Mexico never enacted the WES, so `Treat:Post` here is **not** a WES effect — it
measures any **common, gender-specific post-2018 shift** in women's PSTS self-employment
that also hits Canada. A *non-zero* `Treat:Post` is exactly why the cross-country **DDD**
is needed rather than a single-country Canada DiD: the DDD differences this common shock
away (via `female:Post`). Read this as a diagnostic on the common trend, **not** as a
placebo that has to come out null.

In [ ]:
# Control DiD (Mexico) — same specification as the Canadian DiD
# Mirrors WESEffectsDiD/Main.ipynb exactly: outcome log(SelfEmployedPSTS / LaborForce),
# Treat = female, Treat:Post is the DiD estimate, control = unemployment rate,
# two-way FE C(state)+C(year), SEs clustered by state, full 2014-2024 period.
did = df.dropna(subset=['log_rate', 'labor_force', 'UnemploymentRate']).copy()
did = did[np.isfinite(did['log_rate'])].copy()
did['year']  = did['year'].astype(int)
did['Treat'] = did['female']                      # women = treated group
did['Post']  = (did['year'] >= 2018).astype(int)  # WES enactment cutoff (2018)

did_formula = ('log_rate ~ Treat + Treat:Post + UnemploymentRate '
               '+ C(state) + C(year)')
res_did = smf.ols(did_formula, data=did).fit(
    cov_type='cluster', cov_kwds={'groups': did['state']})
print(res_did.summary())
print('\n--- Mexico control DiD: treatment effect (Treat:Post) ---')
print('coef =', round(res_did.params['Treat:Post'], 4),
      '| std err =', round(res_did.bse['Treat:Post'], 4),
      '| p =', round(res_did.pvalues['Treat:Post'], 4),
      '| n =', int(res_did.nobs), '| states:', did['state'].nunique())

# Styled result graphic (blue header banner + unchanged summary body).
save_result_table(
    'OLS Regression: Mexican Control DiD, Canadian Specification (2014–2024)',
    res_did.summary(), RESULTS / 'Mexico_ControlDiD_OLS_allPSTS.png',
    title_fontsize=13)